# Notebook 1: Data Preprocessing
## E-Commerce Big Data Mining for Customer Intelligence

**Dataset:** Instacart Market Basket data (about 3.4 million orders and 32 million purchased-product rows).

**This notebook covers:** loading, joining, cleaning and validating the data, building the customer features, transforming and scaling them for K-Means, and preparing the basket matrix for FP-Growth.

**Next notebook:** `02_analysis_and_algorithms.ipynb` loads the files saved by this notebook, so run this one first.

**Expected folder layout**
```
project/
├── Dataset/                  <- orders.csv, order_products__prior.csv, products.csv
├── notebooks/                <- this notebook and notebook 2
├── intermediate_data/        <- created here (hand-off files for notebook 2)
├── processed_data/           <- created by notebook 2 (CSV files for the app)
└── app/app.py
```

**Key method decisions (and why)**
- Customer features are built from **prior orders only**, so `total_orders` and `total_products` describe the same set of orders.
- Skewed features are **log-transformed** before scaling so a few extreme customers do not dominate K-Means.
- **Bananas are excluded from rule mining.** They appear in roughly a third of all orders, so nearly every rule would just say "... -> banana".
- The basket matrix is built from a **random sample of 1,000,000 orders**, with support measured against all sampled orders.

## **Step 1: Import Libraries and Set Folders**

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

DATASET_DIR = "../Dataset"
INTERMEDIATE_DIR = "../intermediate_data"

# Folder where the results of this notebook are saved for notebook 2
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

## **Step 2: Load orders.csv**

In [2]:
# Load orders.csv
orders = pd.read_csv(f"{DATASET_DIR}/orders.csv")

# Show first 5 rows
print(orders.head(5))

# Show schema
print(orders.dtypes)

# Count total orders
print("Total orders:", len(orders))

   order_id  user_id eval_set  order_number  order_dow  order_hour_of_day  \
0   2539329        1    prior             1          2                  8   
1   2398795        1    prior             2          3                  7   
2    473747        1    prior             3          3                 12   
3   2254736        1    prior             4          4                  7   
4    431534        1    prior             5          4                 15   

   days_since_prior_order  
0                     NaN  
1                    15.0  
2                    21.0  
3                    29.0  
4                    28.0  
order_id                    int64
user_id                     int64
eval_set                   object
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object
Total orders: 3421083


In [3]:
# The customer features must be built ONLY from "prior" orders.
# orders.csv also contains each customer's last order (train/test set),
# but the products of that order are NOT in order_products__prior.csv.
# Counting it would make total_orders inconsistent with total_products.
print(orders["eval_set"].value_counts())

orders_prior = orders[orders["eval_set"] == "prior"].copy()

print("\nPrior orders:", len(orders_prior))

eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64

Prior orders: 3214874


## **Step 3: Load order_products__prior.csv**

In [4]:
# Load order_products__prior.csv
order_products = pd.read_csv(f"{DATASET_DIR}/order_products__prior.csv")

# Show first 5 records
print(order_products.head(5))

# Show schema
print(order_products.dtypes)

# Count total records
print("Total product-order records:", len(order_products))

   order_id  product_id  add_to_cart_order  reordered
0         2       33120                  1          1
1         2       28985                  2          1
2         2        9327                  3          0
3         2       45918                  4          1
4         2       30035                  5          0
order_id             int64
product_id           int64
add_to_cart_order    int64
reordered            int64
dtype: object
Total product-order records: 32434489


## **Step 4: Load products.csv**

In [5]:
# Load products.csv
products = pd.read_csv(f"{DATASET_DIR}/products.csv")

# Show first 5 products
print(products.head(5))

# Show schema
print(products.dtypes)

# Count products
print("Total products:", len(products))

   product_id                                       product_name  aisle_id  \
0           1                         Chocolate Sandwich Cookies        61   
1           2                                   All-Seasons Salt       104   
2           3               Robust Golden Unsweetened Oolong Tea        94   
3           4  Smart Ones Classic Favorites Mini Rigatoni Wit...        38   
4           5                          Green Chile Anytime Sauce         5   

   department_id  
0             19  
1             13  
2              7  
3              1  
4             13  
product_id        int64
product_name     object
aisle_id          int64
department_id     int64
dtype: object
Total products: 49688


## **Step 5: Join Orders + Products**

In [6]:
# Join orders with order_products
customer_products = pd.merge(
    orders,
    order_products,
    on="order_id",
    how="inner"
)

# Show the result
print(customer_products.head(5))

   order_id  user_id eval_set  order_number  order_dow  order_hour_of_day  \
0   2539329        1    prior             1          2                  8   
1   2539329        1    prior             1          2                  8   
2   2539329        1    prior             1          2                  8   
3   2539329        1    prior             1          2                  8   
4   2539329        1    prior             1          2                  8   

   days_since_prior_order  product_id  add_to_cart_order  reordered  
0                     NaN         196                  1          0  
1                     NaN       14084                  2          0  
2                     NaN       12427                  3          0  
3                     NaN       26088                  4          0  
4                     NaN       26405                  5          0  


In [7]:
print("Columns:", customer_products.columns.tolist())

Columns: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'product_id', 'add_to_cart_order', 'reordered']


In [8]:
# Select only the columns we need
customer_products = customer_products[[
    "user_id",
    "order_id",
    "product_id",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "reordered"
]]

print(customer_products.head(5))

   user_id  order_id  product_id  order_number  order_dow  order_hour_of_day  \
0        1   2539329         196             1          2                  8   
1        1   2539329       14084             1          2                  8   
2        1   2539329       12427             1          2                  8   
3        1   2539329       26088             1          2                  8   
4        1   2539329       26405             1          2                  8   

   reordered  
0          0  
1          0  
2          0  
3          0  
4          0  


## **Step 6: Add Product Names**

In [9]:
customer_products = pd.merge(
    customer_products,
    products,
    on="product_id",
    how="left"
)

print(customer_products.head(5))

   user_id  order_id  product_id  order_number  order_dow  order_hour_of_day  \
0        1   2539329         196             1          2                  8   
1        1   2539329       14084             1          2                  8   
2        1   2539329       12427             1          2                  8   
3        1   2539329       26088             1          2                  8   
4        1   2539329       26405             1          2                  8   

   reordered                             product_name  aisle_id  department_id  
0          0                                     Soda        77              7  
1          0  Organic Unsweetened Vanilla Almond Milk        91             16  
2          0                      Original Beef Jerky        23             19  
3          0               Aged White Cheddar Popcorn        23             19  
4          0         XL Pick-A-Size Paper Towel Rolls        54             17  


In [10]:
# Select only the columns we need
customer_products = customer_products[[
    "user_id",
    "order_id",
    "product_id",
    "product_name",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "reordered"
]]

print(customer_products.head(5))

   user_id  order_id  product_id                             product_name  \
0        1   2539329         196                                     Soda   
1        1   2539329       14084  Organic Unsweetened Vanilla Almond Milk   
2        1   2539329       12427                      Original Beef Jerky   
3        1   2539329       26088               Aged White Cheddar Popcorn   
4        1   2539329       26405         XL Pick-A-Size Paper Towel Rolls   

   order_number  order_dow  order_hour_of_day  reordered  
0             1          2                  8          0  
1             1          2                  8          0  
2             1          2                  8          0  
3             1          2                  8          0  
4             1          2                  8          0  


## **Step 7: Missing Value Analysis**

In [11]:
# Check missing values in each column
missing_values = customer_products.isnull().sum()

print(missing_values)

user_id              0
order_id             0
product_id           0
product_name         0
order_number         0
order_dow            0
order_hour_of_day    0
reordered            0
dtype: int64


## **Step 8: Check Duplicate Records**

In [12]:
# Count total records
total_records = len(customer_products)

print("Total records:", total_records)

Total records: 32434489


In [13]:
# Count unique records
unique_records = len(customer_products.drop_duplicates())

print("Unique records:", unique_records)

Unique records: 32434489


In [14]:
# Duplicate records
print("Duplicate records:", total_records - unique_records)

Duplicate records: 0


## **Step 9: Check Invalid Values**

In [15]:
# Check order_dow (day of week) should be between 0 and 6.
# NOTE: the dataset does not document which number is which day
# (see Step 3 of notebook 2).
print(np.sort(orders["order_dow"].unique()))

[0 1 2 3 4 5 6]


In [16]:
# Check order_hour_of_day should be between 0 and 23
print(np.sort(orders["order_hour_of_day"].unique()))

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]


In [17]:
# Check reordered column should be either 0 or 1
print(order_products["reordered"].unique())

[1 0]


In [18]:
# Check order_number
print(np.sort(orders["order_number"].unique())[:10])

[ 1  2  3  4  5  6  7  8  9 10]


## **Step 10: Create Customer Features**

### Feature 1: Number of Orders
- How many orders did each customer make?

In [19]:
# Count total orders placed by each customer
customer_orders = orders_prior.groupby("user_id").agg(
    {"order_id": "count"}
).reset_index()

# Rename the column to "total_orders"
customer_orders = customer_orders.rename(
    columns={"order_id": "total_orders"}
)

print(customer_orders.head(10))

   user_id  total_orders
0        1            10
1        2            14
2        3            12
3        4             5
4        5             4
5        6             3
6        7            20
7        8             3
8        9             3
9       10             5


### Feature 2: Total Products Purchased
- How many products did each customer purchase in total?

In [20]:
# Count total products purchased by each customer
customer_products_count = customer_products.groupby("user_id").agg(
    {"product_id": "count"}
).reset_index()

# Rename the column to "total_products"
customer_products_count = customer_products_count.rename(
    columns={"product_id": "total_products"}
)

print(customer_products_count.head(10))

   user_id  total_products
0        1              59
1        2             195
2        3              88
3        4              18
4        5              37
5        6              14
6        7             206
7        8              49
8        9              76
9       10             143


### Feature 3: Average Products Per Order
- On average, how many products does a customer purchase in one order?

In [21]:
# Join customer_orders and customer_products_count to create the customer features
customer_features = pd.merge(
    customer_orders,
    customer_products_count,
    on="user_id",
    how="inner"
)

# Calculate average products per order for each customer
customer_features["avg_products_per_order"] = (
    customer_features["total_products"] / customer_features["total_orders"]
)

print(customer_features.head(10))

   user_id  total_orders  total_products  avg_products_per_order
0        1            10              59                5.900000
1        2            14             195               13.928571
2        3            12              88                7.333333
3        4             5              18                3.600000
4        5             4              37                9.250000
5        6             3              14                4.666667
6        7            20             206               10.300000
7        8             3              49               16.333333
8        9             3              76               25.333333
9       10             5             143               28.600000


### Feature 4: Reorder Rate
- What percentage of a customer's purchased products were reordered?

In [22]:
# Calculate Reorder Rate = Reordered Products / Total Products
customer_reorder = customer_products.groupby("user_id").agg(
    reorder_rate=("reordered", "mean")
).reset_index()

print(customer_reorder.head(10))

   user_id  reorder_rate
0        1      0.694915
1        2      0.476923
2        3      0.625000
3        4      0.055556
4        5      0.378378
5        6      0.142857
6        7      0.669903
7        8      0.265306
8        9      0.236842
9       10      0.342657


In [23]:
# Join customer_features with customer_reorder to add reorder_rate
customer_features = pd.merge(
    customer_features,
    customer_reorder,
    on="user_id",
    how="inner"
)

print(customer_features.head(10))

   user_id  total_orders  total_products  avg_products_per_order  reorder_rate
0        1            10              59                5.900000      0.694915
1        2            14             195               13.928571      0.476923
2        3            12              88                7.333333      0.625000
3        4             5              18                3.600000      0.055556
4        5             4              37                9.250000      0.378378
5        6             3              14                4.666667      0.142857
6        7            20             206               10.300000      0.669903
7        8             3              49               16.333333      0.265306
8        9             3              76               25.333333      0.236842
9       10             5             143               28.600000      0.342657


### Feature 5: Average Days Between Orders
- How frequently does a customer return to shop?

> Note: `days_since_prior_order` is capped at 30 days in the source data, so averages are slightly underestimated for infrequent customers.

In [24]:
# Calculate average days between orders for each customer
customer_days = orders_prior.groupby("user_id").agg(
    avg_days_between_orders=("days_since_prior_order", "mean")
).reset_index()

print(customer_days.head(10))

   user_id  avg_days_between_orders
0        1                19.555556
1        2                15.230769
2        3                12.090909
3        4                13.750000
4        5                13.333333
5        6                 9.000000
6        7                10.684211
7        8                30.000000
8        9                18.000000
9       10                19.750000


In [25]:
# Join customer_features with customer_days to add avg_days_between_orders
customer_features = pd.merge(
    customer_features,
    customer_days,
    on="user_id",
    how="inner"
)

print(customer_features.head(10))

   user_id  total_orders  total_products  avg_products_per_order  \
0        1            10              59                5.900000   
1        2            14             195               13.928571   
2        3            12              88                7.333333   
3        4             5              18                3.600000   
4        5             4              37                9.250000   
5        6             3              14                4.666667   
6        7            20             206               10.300000   
7        8             3              49               16.333333   
8        9             3              76               25.333333   
9       10             5             143               28.600000   

   reorder_rate  avg_days_between_orders  
0      0.694915                19.555556  
1      0.476923                15.230769  
2      0.625000                12.090909  
3      0.055556                13.750000  
4      0.378378                13.33

In [26]:
# Check the final customer_features DataFrame
print("Total customers:", len(customer_features))
print("Total features:", len(customer_features.columns))

# Sanity check: every prior order should be counted exactly once
print("Orders counted in features:", customer_features["total_orders"].sum())
print("Prior orders in orders.csv:", len(orders_prior))

print()
print(customer_features.dtypes)

Total customers: 206209
Total features: 6
Orders counted in features: 3214874
Prior orders in orders.csv: 3214874

user_id                      int64
total_orders                 int64
total_products               int64
avg_products_per_order     float64
reorder_rate               float64
avg_days_between_orders    float64
dtype: object


## **Step 11: Prepare Features for K-Means**

Order counts and basket sizes are strongly right-skewed (a few very heavy shoppers). Without a transformation, those customers dominate the distance calculation. We apply `log1p` to the skewed features and check the correlation between features.

In [27]:
features = [
    "total_orders",
    "total_products",
    "avg_products_per_order",
    "reorder_rate",
    "avg_days_between_orders"
]

skewed_features = ["total_orders", "total_products", "avg_products_per_order"]

# Safety: K-Means cannot handle missing values
print("Missing values:\n", customer_features[features].isnull().sum())
customer_features["avg_days_between_orders"] = customer_features[
    "avg_days_between_orders"
].fillna(customer_features["avg_days_between_orders"].median())

print("\nSkewness BEFORE log transform:")
print(customer_features[features].skew().round(2))

# Log-transform the skewed features (only for clustering; the original
# values are kept in customer_features for reporting)
customer_model = customer_features[features].copy()

for col in skewed_features:
    customer_model[col] = np.log1p(customer_model[col])

print("\nSkewness AFTER log transform:")
print(customer_model.skew().round(2))

print("\nFeature correlation (original values):")
print(customer_features[features].corr().round(2))

Missing values:
 total_orders               0
total_products             0
avg_products_per_order     0
reorder_rate               0
avg_days_between_orders    0
dtype: int64

Skewness BEFORE log transform:
total_orders               2.40
total_products             3.18
avg_products_per_order     1.23
reorder_rate               0.03
avg_days_between_orders    0.34
dtype: float64

Skewness AFTER log transform:
total_orders               0.56
total_products             0.05
avg_products_per_order    -0.33
reorder_rate               0.03
avg_days_between_orders    0.34
dtype: float64

Feature correlation (original values):
                         total_orders  total_products  avg_products_per_order  \
total_orders                     1.00            0.81                    0.02   
total_products                   0.81            1.00                    0.42   
avg_products_per_order           0.02            0.42                    1.00   
reorder_rate                     0.64           

## **Step 12: Standardize the Features**

In [28]:
# Scale the (log-transformed) features using StandardScaler
scaler = StandardScaler(
    with_mean=True,
    with_std=True
)

scaled_features = scaler.fit_transform(customer_model[features])

print("Scaled matrix shape:", scaled_features.shape)
print(scaled_features[:5].round(3))

Scaled matrix shape: (206209, 5)
[[-0.07  -0.336 -0.578  1.238  0.612]
 [ 0.317  0.733  0.821  0.211  0.003]
 [ 0.138  0.02  -0.236  0.909 -0.439]
 [-0.827 -1.375 -1.313 -1.776 -0.205]
 [-1.055 -0.749  0.139 -0.254 -0.264]]


## **Step 13: Select Products for Market Basket Analysis**

First we count how often each product was purchased.

Bananas (`Banana`, `Bag of Organic Bananas`) appear in roughly a third of all orders. If they stay in, almost every rule ends in "... -> banana", which is true but useless for recommendations. We treat them as **staples** and exclude them from rule mining, then use the **top 150** remaining products so more products get recommendations.

In [29]:
# Count how often each product was purchased (all prior orders)
product_counts = customer_products["product_name"].value_counts()

print("Total unique products:", len(product_counts))
print("\nTop 20 most purchased products:")
print(product_counts.head(20))

Total unique products: 49677

Top 20 most purchased products:
product_name
Banana                      472565
Bag of Organic Bananas      379450
Organic Strawberries        264683
Organic Baby Spinach        241921
Organic Hass Avocado        213584
Organic Avocado             176815
Large Lemon                 152657
Strawberries                142951
Limes                       140627
Organic Whole Milk          137905
Organic Raspberries         137057
Organic Yellow Onion        113426
Organic Garlic              109778
Organic Zucchini            104823
Organic Blueberries         100060
Cucumber Kirby               97315
Organic Fuji Apple           89632
Organic Lemon                87746
Apple Honeycrisp Organic     85020
Organic Grape Tomatoes       84255
Name: count, dtype: int64


In [30]:
STAPLE_PRODUCTS = ["Banana", "Bag of Organic Bananas"]
TOP_N_PRODUCTS = 150
SAMPLE_ORDERS = 1_000_000

# Most purchased products, without the staples
candidate_products = product_counts.drop(labels=STAPLE_PRODUCTS, errors="ignore")

top_products = candidate_products.head(TOP_N_PRODUCTS).index

print("Staples excluded:", STAPLE_PRODUCTS)
print("Number of selected products:", len(top_products))
print("\nTop 10 selected products:")
print(list(top_products[:10]))

Staples excluded: ['Banana', 'Bag of Organic Bananas']
Number of selected products: 150

Top 10 selected products:
['Organic Strawberries', 'Organic Baby Spinach', 'Organic Hass Avocado', 'Organic Avocado', 'Large Lemon', 'Strawberries', 'Limes', 'Organic Whole Milk', 'Organic Raspberries', 'Organic Yellow Onion']


## **Step 14: Sample Orders and Filter the Transaction Data**

A random sample of 1,000,000 prior orders keeps memory usage manageable; support, confidence and lift estimated from that many random orders are very close to the full-data values. Orders that contain none of the selected products are **kept as empty baskets**, so support is measured against all orders and is not inflated.

In [31]:
sampled_orders = orders_prior["order_id"].sample(
    n=min(SAMPLE_ORDERS, len(orders_prior)),
    random_state=42
)

market_basket_data = customer_products[
    customer_products["order_id"].isin(sampled_orders)
    & customer_products["product_name"].isin(top_products)
]

print("Sampled orders:", len(sampled_orders))
print("Original transaction rows:", len(customer_products))
print("Filtered transaction rows:", len(market_basket_data))
print("Unique products:", market_basket_data["product_name"].nunique())

Sampled orders: 1000000
Original transaction rows: 32434489
Filtered transaction rows: 2511173
Unique products: 150


## **Step 15: Create the Basket Matrix**

In [32]:
# Basket matrix: one row per sampled order, one column per product
basket = (
    market_basket_data
    .groupby(["order_id", "product_name"])["product_id"]
    .count()
    .unstack(fill_value=0)
    .gt(0)
)

# Add the orders that contain none of the selected products (all False)
basket = basket.reindex(sampled_orders.values, fill_value=False).astype(bool)

print("Basket shape:", basket.shape)
print("Average selected products per order:", round(basket.sum(axis=1).mean(), 2))

Basket shape: (1000000, 150)
Average selected products per order: 2.51


## **Step 16: Save the Preprocessed Data for Notebook 2**

Notebook 2 should not have to reload and join 32 million rows, so we save only what it needs. Pickle files (`.pkl`) keep the column types exactly as they are and need no extra library.

| File | Used for |
| --- | --- |
| `customer_features.pkl` | customer features (original values) |
| `scaled_features.pkl` | log-transformed and standardized features for K-Means |
| `basket.pkl` | True/False basket matrix for FP-Growth |
| `product_counts.pkl` | purchase count of every product |
| `user_product_counts.pkl` | how many times each customer bought each product |
| `orders_prior.pkl` | prior orders (day of week and hour analysis) |
| `products.pkl` | product id to product name lookup |

In [33]:
# 1. Customer features (original values, missing days filled in Step 11)
customer_features.to_pickle(f"{INTERMEDIATE_DIR}/customer_features.pkl")

# 2. Scaled features, with user_id so notebook 2 can check the row order
scaled_df = pd.DataFrame(scaled_features, columns=features)
scaled_df.insert(0, "user_id", customer_features["user_id"].values)
scaled_df.to_pickle(f"{INTERMEDIATE_DIR}/scaled_features.pkl")

# 3. Basket matrix for FP-Growth
basket.to_pickle(f"{INTERMEDIATE_DIR}/basket.pkl")

# 4. Purchase count of every product
(
    product_counts
    .rename_axis("product_name")
    .reset_index(name="purchase_count")
    .to_pickle(f"{INTERMEDIATE_DIR}/product_counts.pkl")
)

# 5. How many times each customer bought each product
#    (small summary of the 32-million-row table, used for top products per segment)
user_product_counts = (
    customer_products
    .groupby(["user_id", "product_id"])
    .size()
    .rename("purchase_count")
    .reset_index()
)
user_product_counts.to_pickle(f"{INTERMEDIATE_DIR}/user_product_counts.pkl")

# 6. Prior orders (only the columns needed for the day/hour analysis)
orders_prior[["order_id", "user_id", "order_dow", "order_hour_of_day"]].to_pickle(
    f"{INTERMEDIATE_DIR}/orders_prior.pkl"
)

# 7. Product lookup
products[["product_id", "product_name"]].to_pickle(f"{INTERMEDIATE_DIR}/products.pkl")

print("Saved files in", INTERMEDIATE_DIR)
for name in sorted(os.listdir(INTERMEDIATE_DIR)):
    size_mb = os.path.getsize(os.path.join(INTERMEDIATE_DIR, name)) / 1_000_000
    print(f"  {name:28s} {size_mb:8.1f} MB")

Saved files in ../intermediate_data
  basket.pkl                      158.0 MB
  customer_features.pkl             9.9 MB
  orders_prior.pkl                128.6 MB
  product_counts.pkl                2.1 MB
  products.pkl                      2.1 MB
  scaled_features.pkl               9.9 MB
  user_product_counts.pkl         319.4 MB


## **Done**

Data preprocessing is complete. Now open and run **`02_analysis_and_algorithms.ipynb`**.